In [2]:
# 라이브러리 실행 확인용
import pandas as pd
import os

In [ ]:
# details_kr_part1.parquet csv로 변환
import pandas as pd
import os

def view_and_save_with_absolute_path(input_path, output_path):
    """
    지정된 절대 경로의 Parquet 파일을 읽어 데이터를 확인하고 CSV로 저장하는 함수입니다.
    """
    # 1. 파일 존재 여부 확인
    if not os.path.exists(input_path):
        print(f"오류: 지정한 경로에 파일이 존재하지 않습니다. 경로를 다시 확인해주세요.\n{input_path}")
        return None
        
    try:
        print("데이터를 불러오는 중입니다. 잠시만 기다려주세요.")
        
        # 2. 데이터 불러오기
        df = pd.read_parquet(input_path, engine='pyarrow')
        
        # 3. 전체 열을 볼 수 있도록 옵션 설정
        pd.set_option('display.max_columns', None)
        
        # 4. 데이터 미리보기
        print("\n[데이터 미리보기]")
        display(df.head())
        print(f"\n총 데이터 크기: {df.shape[0]}행, {df.shape[1]}열")
        
        # 5. CSV 파일로 저장
        print("\n데이터를 CSV 파일로 저장하는 중입니다...")
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        
        print(f"저장 완료! 아래 경로에 CSV 파일이 생성되었습니다.\n{output_path}")
        
        return df
        
    except Exception as e:
        print(f"파일을 처리하는 도중 문제가 발생했습니다: {e}")
        return None

# 알려주신 원본 파일의 절대 경로 (문자열 앞에 r을 붙여 백슬래시 오류 방지)
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details_kr_part1.parquet"

# 저장할 CSV 파일의 절대 경로 (동일한 폴더, 확장자만 csv로 변경)
csv_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details_kr_part1.csv"

# 함수 실행
kr_data = view_and_save_with_absolute_path(file_path, csv_path)

데이터를 불러오는 중입니다. 잠시만 기다려주세요.

[데이터 미리보기]


,metadata,info
0,"{'dataVersion': '2', 'matchId': 'KR_7732890911...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
1,"{'dataVersion': '2', 'matchId': 'KR_8006003717...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
2,"{'dataVersion': '2', 'matchId': 'KR_7535288608...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
3,"{'dataVersion': '2', 'matchId': 'KR_8068731228...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
4,"{'dataVersion': '2', 'matchId': 'KR_7023419356...","{'endOfGameResult': 'GameComplete', 'gameCreat..."



총 데이터 크기: 23523행, 2열

데이터를 CSV 파일로 저장하는 중입니다...
저장 완료! 아래 경로에 CSV 파일이 생성되었습니다.
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details_kr_part1.csv


In [ ]:
#위 결과 csv 파일에서 회의로 통해 나눈 컬럼만 뽑기
import pandas as pd
import numpy as np
import os

def process_full_parquet_to_csv_fixed(input_path, output_path, target_cols):
    if not os.path.exists(input_path):
        print(f"오류: 지정한 경로에 파일이 존재하지 않습니다.\n{input_path}")
        return None
        
    try:
        print("1. Parquet 파일 로드 중... (시간이 조금 걸릴 수 있습니다)")
        df_raw = pd.read_parquet(input_path, engine='pyarrow')
        total_matches = len(df_raw)
        print(f" -> 총 {total_matches}개의 매치(게임) 데이터를 성공적으로 불러왔습니다.\n")
        
        processed_rows = []
        print("2. 데이터 변환(평탄화) 작업을 시작합니다...")
        
        # 전체 매치를 한 줄씩 순회
        for index, row in df_raw.iterrows():
            if (index + 1) % 1000 == 0:
                print(f" -> 진행 중: {index + 1} / {total_matches} 매치 변환 완료")
                
            # 라이엇 API 구조 반영: metadata와 info 분리 확인
            # 데이터프레임의 최상단 컬럼이거나, 내부에 들어있을 경우 모두 대비
            metadata = row.get('metadata', {})
            info = row.get('info', {})
            
            # 자료형 안전 변환
            if pd.notnull(metadata) and not isinstance(metadata, dict):
                try: metadata = dict(metadata)
                except: metadata = {}
            if pd.notnull(info) and not isinstance(info, dict):
                try: info = dict(info)
                except: info = {}

            # 공통 데이터 추출
            common_info = {
                'dataVersion': metadata.get('dataVersion', row.get('dataVersion')),
                'matchId': metadata.get('matchId', row.get('matchId')),
                'endOfGameResult': info.get('endOfGameResult', row.get('endOfGameResult')),
                'gameCreation': info.get('gameCreation', row.get('gameCreation')),
                'gameDuration': info.get('gameDuration', row.get('gameDuration')),
                'gameVersion': info.get('gameVersion', row.get('gameVersion')),
                'queueId': info.get('queueId', row.get('queueId'))
            }
            
            # 참가자(participants) 정보는 info 안에 존재함
            participants = info.get('participants', row.get('participants', []))
            
            # numpy 배열 형태라면 리스트로 변환
            if isinstance(participants, np.ndarray):
                participants = participants.tolist()
                
            if not isinstance(participants, list):
                continue
                
            # 10명의 플레이어를 순회
            for p in participants:
                # pyarrow struct 데이터를 dict로 강제 변환
                if not isinstance(p, dict):
                    try: p = dict(p)
                    except: continue
                    
                player_row = {}
                challenges = p.get('challenges', {})
                if not isinstance(challenges, dict):
                    try: challenges = dict(challenges)
                    except: challenges = {}
                
                # 요청하신 타겟 컬럼에 맞춰 데이터 매핑
                for col in target_cols:
                    if col in common_info:
                        player_row[col] = common_info[col]
                    elif col.startswith('ch_'):
                        actual_key = col[3:]  # 'ch_' 제외한 원래 이름
                        player_row[col] = challenges.get(actual_key, None)
                    else:
                        player_row[col] = p.get(col, None)
                        
                processed_rows.append(player_row)
                
        print("\n3. 변환된 데이터를 DataFrame으로 조립하는 중...")
        final_df = pd.DataFrame(processed_rows, columns=target_cols)
        print(f" -> 최종 데이터 크기: {final_df.shape[0]}행(플레이어 수), {final_df.shape[1]}열")
        
        if final_df.shape[0] == 0:
            print("\n[경고] 추출된 데이터가 여전히 0행입니다.")
            print("데이터의 실제 컬럼명 구조 확인용 출력:")
            print(df_raw.columns.tolist())
            return final_df
        
        print(f"\n4. CSV 파일로 저장하는 중입니다...")
        final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f" -> 저장 완료! 아래 경로에서 확인해주세요.\n{output_path}")
        
        return final_df
        
    except Exception as e:
        print(f"처리 도중 오류가 발생했습니다: {e}")
        return None

# ==========================================
# 실행을 위한 설정 영역
# ==========================================

target_columns = [
    "assists", "baronKills", "bountyLevel", "ch_abilityUses", "ch_acesBefore15Minutes",
    "ch_alliedJungleMonsterKills", "ch_baronBuffGoldAdvantageOverThreshold", "ch_baronTakedowns",
    "ch_bountyGold", "ch_buffsStolen", "ch_completeSupportQuestInTime", "ch_controlWardTimeCoverageInRiverOrEnemyHalf",
    "ch_controlWardsPlaced", "ch_damagePerMinute", "ch_damageTakenOnTeamPercentage", "ch_deathsByEnemyChamps",
    "ch_dragonTakedowns", "ch_earlyLaningPhaseGoldExpAdvantage", "ch_effectiveHealAndShielding", "ch_enemyChampionImmobilizations",
    "ch_enemyJungleMonsterKills", "ch_epicMonsterKillsWithin30SecondsOfSpawn", "ch_epicMonsterSteals", "ch_goldPerMinute",
    "ch_hadAfkTeammate", "ch_highestChampionDamage", "ch_highestCrowdControlScore", "ch_highestWardKills",
    "ch_immobilizeAndKillWithAlly", "ch_initialBuffCount", "ch_initialCrabCount", "ch_jungleCsBefore10Minutes",
    "ch_junglerKillsEarlyJungle", "ch_junglerTakedownsNearDamagedEpicMonster", "ch_kTurretsDestroyedBeforePlatesFall",
    "ch_kda", "ch_killParticipation", "ch_killingSprees", "ch_killsNearEnemyTurret", "ch_killsOnLanersEarlyJungleAsJungler",
    "ch_killsOnOtherLanesEarlyJungleAsLaner", "ch_killsUnderOwnTurret", "ch_knockEnemyIntoTeamAndKill", "ch_landSkillShotsEarlyGame",
    "ch_laneMinionsFirst10Minutes", "ch_laningPhaseGoldExpAdvantage", "ch_legendaryCount", "ch_maxCsAdvantageOnLaneOpponent",
    "ch_maxKillDeficit", "ch_maxLevelLeadLaneOpponent", "ch_mejaisFullStackInTime", "ch_moreEnemyJungleThanOpponent",
    "ch_multikills", "ch_outerTurretExecutesBefore10Minutes", "ch_playedChampSelectPosition", "ch_quickFirstTurret",
    "ch_quickSoloKills", "ch_riftHeraldTakedowns", "ch_saveAllyFromDeath", "ch_scuttleCrabKills", "ch_soloKills",
    "ch_stealthWardsPlaced", "ch_takedowns", "ch_takedownsAfterGainingLevelAdvantage", "ch_takedownsBeforeJungleMinionSpawn",
    "ch_takedownsFirstXMinutes", "ch_takedownsInEnemyFountain", "ch_teamBaronKills", "ch_teamDamagePercentage",
    "ch_teamElderDragonKills", "ch_teamRiftHeraldKills", "ch_turretPlatesTaken", "ch_turretTakedowns", "ch_turretsTakenWithRiftHerald",
    "ch_unseenRecalls", "ch_visionScoreAdvantageLaneOpponent", "ch_visionScorePerMinute", "ch_wardTakedowns",
    "ch_wardTakedownsBefore20M", "champExperience", "champLevel", "championName", "championTransform",
    "damageDealtToBuildings", "damageDealtToEpicMonsters", "damageDealtToObjectives", "damageDealtToTurrets",
    "damageSelfMitigated", "dataVersion", "deaths", "detectorWardsPlaced", "doubleKills", "eligibleForProgression",
    "endOfGameResult", "firstBloodAssist", "firstBloodKill", "firstTowerAssist", "firstTowerKill", "gameCreation",
    "gameDuration", "gameEndedInEarlySurrender", "gameEndedInSurrender", "gameVersion", "goldEarned", "goldSpent",
    "individualPosition", "inhibitorTakedowns", "inhibitorsLost", "itemsPurchased", "killingSprees", "kills",
    "magicDamageDealt", "magicDamageDealtToChampions", "magicDamageTaken", "matchId", "neutralMinionsKilled",
    "objectivesStolen", "objectivesStolenAssists", "participantId", "perks", "physicalDamageDealt", "physicalDamageDealtToChampions",
    "physicalDamageTaken", "queueId", "summonerId", "summonerLevel", "teamEarlySurrendered", "teamId", "teamPosition",
    "timeCCingOthers", "timePlayed", "totalAllyJungleMinionsKilled", "totalDamageDealt", "totalDamageDealtToChampions",
    "totalDamageShieldedOnTeammates", "totalDamageTaken", "totalEnemyJungleMinionsKilled", "totalHeal", "totalHealsOnTeammates",
    "totalMinionsKilled", "totalTimeCCDealt", "totalTimeSpentDead", "totalUnitsHealed", "trueDamageDealt",
    "trueDamageDealtToChampions", "trueDamageTaken", "turretKills", "turretTakedowns", "turretsLost", "visionScore",
    "visionWardsBoughtInGame", "wardsKilled", "wardsPlaced", "win"
]

file_path_in = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details_kr_part1.parquet"
file_path_out = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details_kr_part1_flattened.csv"

final_kr_data = process_full_parquet_to_csv_fixed(file_path_in, file_path_out, target_columns)

if final_kr_data is not None and final_kr_data.shape[0] > 0:
    pd.set_option('display.max_columns', 20)
    display(final_kr_data.head())

1. Parquet 파일 로드 중... (시간이 조금 걸릴 수 있습니다)
 -> 총 23523개의 매치(게임) 데이터를 성공적으로 불러왔습니다.

2. 데이터 변환(평탄화) 작업을 시작합니다...
 -> 진행 중: 1000 / 23523 매치 변환 완료
 -> 진행 중: 2000 / 23523 매치 변환 완료
 -> 진행 중: 3000 / 23523 매치 변환 완료
 -> 진행 중: 4000 / 23523 매치 변환 완료
 -> 진행 중: 5000 / 23523 매치 변환 완료
 -> 진행 중: 6000 / 23523 매치 변환 완료
 -> 진행 중: 7000 / 23523 매치 변환 완료
 -> 진행 중: 8000 / 23523 매치 변환 완료
 -> 진행 중: 9000 / 23523 매치 변환 완료
 -> 진행 중: 10000 / 23523 매치 변환 완료
 -> 진행 중: 11000 / 23523 매치 변환 완료
 -> 진행 중: 12000 / 23523 매치 변환 완료
 -> 진행 중: 13000 / 23523 매치 변환 완료
 -> 진행 중: 14000 / 23523 매치 변환 완료
 -> 진행 중: 15000 / 23523 매치 변환 완료
 -> 진행 중: 16000 / 23523 매치 변환 완료
 -> 진행 중: 17000 / 23523 매치 변환 완료
 -> 진행 중: 18000 / 23523 매치 변환 완료
 -> 진행 중: 19000 / 23523 매치 변환 완료
 -> 진행 중: 20000 / 23523 매치 변환 완료
 -> 진행 중: 21000 / 23523 매치 변환 완료
 -> 진행 중: 22000 / 23523 매치 변환 완료
 -> 진행 중: 23000 / 23523 매치 변환 완료

3. 변환된 데이터를 DataFrame으로 조립하는 중...
 -> 최종 데이터 크기: 235080행(플레이어 수), 154열

4. CSV 파일로 저장하는 중입니다...
 -> 저장 완료! 아래 경로에서 확인해주세요.
D:\바탕화면\데이터 분석\프

,assists,baronKills,bountyLevel,ch_abilityUses,ch_acesBefore15Minutes,ch_alliedJungleMonsterKills,ch_baronBuffGoldAdvantageOverThreshold,ch_baronTakedowns,ch_bountyGold,ch_buffsStolen,...,trueDamageDealtToChampions,trueDamageTaken,turretKills,turretTakedowns,turretsLost,visionScore,visionWardsBoughtInGame,wardsKilled,wardsPlaced,win
0,2,0,NaN,110,0,0,NaN,0,0.000000,0,...,0,32,1,1,1,7,0,0,5,False
1,3,0,NaN,254,0,15,NaN,0,469.360535,1,...,84,471,0,0,1,3,1,1,1,False
2,4,0,NaN,115,0,0,NaN,0,637.141968,0,...,0,59,0,1,1,9,0,1,5,False
3,5,0,NaN,60,0,0,NaN,0,359.220337,0,...,0,397,0,0,1,9,0,1,5,False
4,4,0,NaN,97,0,0,NaN,0,0.000000,0,...,299,615,0,0,1,30,7,3,17,False


In [ ]:
# timeline_kr_part1.parquet csv로 변환
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import os
import gc

def extract_all_timeline_frames(input_path, output_path):
    """
    타임라인 파일에서 1분 단위의 플레이어의 모든 상태(위치, 골드, 데미지스탯, 챔피언스탯 등)를
    동적으로 추출하여 CSV로 안전하게 저장합니다.
    """
    if not os.path.exists(input_path):
        print(f"오류: 지정한 경로에 파일이 존재하지 않습니다.\n{input_path}")
        return
        
    try:
        print("1. 타임라인 전체 데이터 추출 준비 중...")
        parquet_file = pq.ParquetFile(input_path)
        total_batches = parquet_file.num_row_groups
        print(f" -> 총 {total_batches}개의 데이터 그룹 발견.\n")
        
        if os.path.exists(output_path):
            os.remove(output_path)
            
        is_first_batch = True
        csv_columns = [] # CSV 컬럼 순서를 고정하기 위한 변수
        
        print("2. 1분 단위 플레이어 [전체] 데이터 추출 및 저장 시작...")
        
        # 메모리 안정을 위해 50게임씩 처리
        for batch_idx, batch in enumerate(parquet_file.iter_batches(batch_size=50)):
            df_chunk = batch.to_pandas()
            processed_rows = []
            
            for index, row in df_chunk.iterrows():
                # 데이터 안전 변환
                metadata = row.get('metadata', {})
                info = row.get('info', {})
                
                if pd.notnull(metadata) and not isinstance(metadata, dict):
                    try: metadata = dict(metadata)
                    except: metadata = {}
                if pd.notnull(info) and not isinstance(info, dict):
                    try: info = dict(info)
                    except: info = {}
                    
                match_id = metadata.get('matchId', 'Unknown')
                frames = info.get('frames', [])
                
                if isinstance(frames, np.ndarray):
                    frames = frames.tolist()
                if not isinstance(frames, list):
                    continue
                
                for frame in frames:
                    if not isinstance(frame, dict):
                        continue
                        
                    timestamp = frame.get('timestamp', 0)
                    minute = timestamp // 60000 
                    
                    participant_frames = frame.get('participantFrames', {})
                    if not isinstance(participant_frames, dict):
                        continue
                        
                    for pid, p_data in participant_frames.items():
                        if not isinstance(p_data, dict):
                            continue
                            
                        # 1) 기본 필수 정보 세팅
                        row_dict = {
                            'matchId': match_id,
                            'minute': minute,
                            'participantId': p_data.get('participantId', pid)
                        }
                        
                        # 2) p_data 내의 '모든' 정보를 동적으로 펼치기(Flatten)
                        for key, value in p_data.items():
                            if key == 'participantId': 
                                continue # 위에서 이미 넣었으므로 패스
                                
                            if isinstance(value, dict):
                                # damageStats, championStats, position 등 내부 딕셔너리 분해
                                for sub_key, sub_val in value.items():
                                    row_dict[f"{key}_{sub_key}"] = sub_val
                            else:
                                # 일반 수치형 데이터 (totalGold, level, xp 등)
                                row_dict[key] = value
                                
                        processed_rows.append(row_dict)
                        
            # 조립된 데이터를 CSV에 저장
            if processed_rows:
                chunk_df = pd.DataFrame(processed_rows)
                
                if is_first_batch:
                    # 첫 번째 배치에서 생성된 컬럼 리스트와 순서를 기억합니다.
                    csv_columns = chunk_df.columns.tolist()
                    chunk_df.to_csv(output_path, mode='w', index=False, encoding='utf-8-sig')
                    is_first_batch = False
                else:
                    # 두 번째 배치부터는 반드시 첫 번째 배치와 동일한 컬럼 순서로 정렬합니다.
                    # (만약 특정 판에서 누락된 열이 있다면 자동으로 빈칸(NaN) 처리하여 표가 밀리는 것을 방지)
                    chunk_df = chunk_df.reindex(columns=csv_columns)
                    chunk_df.to_csv(output_path, mode='a', header=False, index=False, encoding='utf-8-sig')
                    
            # 사용이 끝난 메모리 즉시 반환 (OOM 방지)
            del df_chunk
            del processed_rows
            del chunk_df
            gc.collect()
            
            print(f" -> 진행 상황: {batch_idx + 1} / {total_batches} 청크 추출 및 저장 완료")
            
        print(f"\n3. 모든 추출이 성공적으로 완료되었습니다!\n저장 경로: {output_path}")

    except Exception as e:
        print(f"작업 중 오류 발생: {e}")

# ==========================================
# 실행을 위한 설정 영역
# ==========================================

file_path_in = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\timeline_kr_part1.parquet"

# 이전 파일과 구분하기 위해 이름 뒤에 '_all_data'를 붙였습니다.
file_path_out = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\timeline_kr_part1_all_data.csv"

# 함수 실행
extract_all_timeline_frames(file_path_in, file_path_out)

1. 타임라인 전체 데이터 추출 준비 중...
 -> 총 1개의 데이터 그룹 발견.

2. 1분 단위 플레이어 [전체] 데이터 추출 및 저장 시작...
 -> 진행 상황: 1 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 2 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 3 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 4 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 5 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 6 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 7 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 8 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 9 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 10 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 11 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 12 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 13 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 14 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 15 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 16 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 17 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 18 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 19 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 20 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 21 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 22 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 23 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 24 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 25 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 26 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 27 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 28 / 1 청크 추출 및 저장 완료
 -> 진행 상황: 29 / 1 청크 추출 및 저장

In [ ]:
# timeline.csv 파일에서 회의로 통해 나눈 컬럼만 뽑기
import pandas as pd
import os

def filter_timeline_columns_fixed(input_csv, output_csv):
    """
    점(.) 대신 밑줄(_)을 사용하여 damageStats 컬럼을 정확히 필터링합니다.
    """
    if not os.path.exists(input_csv):
        print(f"오류: '{input_csv}' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return

    print("1. 원본 파일의 컬럼(열) 구조를 분석 중입니다...")
    
    # 0행만 읽어서 전체 컬럼명 파악
    actual_columns = pd.read_csv(input_csv, nrows=0).columns.tolist()
    
    # [수정됨] 점(.) 대신 밑줄(_) 사용 및 필수 키워드 추가
    keep_keywords = [
        "matchId",        # 어떤 게임인지 알아야 하므로 추가
        "minute",         # 게임 시간(분) 추가
        "timestamp",
        "events",
        "participantId",
        "totalGold",
        "level",
        "xp",
        "minionsKilled",
        "jungleMinionsKilled",
        "timeEnemySpentControlled",
        "damageStats_magicDamageDone",
        "damageStats_magicDamageDoneToChampions",
        "damageStats_magicDamageTaken",
        "damageStats_physicalDamageDone",
        "damageStats_physicalDamageDoneToChampions",
        "damageStats_physicalDamageTaken",
        "damageStats_totalDamageDone",
        "damageStats_totalDamageDoneToChampions",
        "damageStats_totalDamageTaken",
        "damageStats_trueDamageDone",
        "damageStats_trueDamageDoneToChampions",
        "damageStats_trueDamageTaken"
    ]
    
    columns_to_keep = []
    
    for col in actual_columns:
        if any(keyword in col for keyword in keep_keywords):
            columns_to_keep.append(col)
            
    print(f" -> 총 {len(actual_columns)}개의 컬럼 중 {len(columns_to_keep)}개의 컬럼만 남깁니다.")
    
    # 제대로 잡혔는지 확인하기 위해 데미지 스탯 일부 출력
    damage_cols_found = [c for c in columns_to_keep if 'damageStats' in c]
    print(f" -> [확인용] 찾은 데미지 스탯 개수: {len(damage_cols_found)}개")
    if damage_cols_found:
        print(f" -> 예시: {damage_cols_found[:3]}")
    
    if os.path.exists(output_csv):
        os.remove(output_csv)
        
    print("\n2. 선택된 컬럼만 추출하여 새로운 CSV로 저장합니다... (메모리 안전 모드)")
    
    is_first = True
    for chunk in pd.read_csv(input_csv, usecols=columns_to_keep, chunksize=10000):
        if is_first:
            chunk.to_csv(output_csv, mode='w', index=False, encoding='utf-8-sig')
            is_first = False
        else:
            chunk.to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8-sig')
            
    print(f"\n3. 깔끔하게 정리가 완료되었습니다! 파일 저장 경로:\n{output_csv}")

# ==========================================
# 실행을 위한 파일 경로 설정
# ==========================================

file_path_in = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\timeline_kr_part1_all_data.csv"
file_path_out = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\timeline_kr_part1_all_data_filtered.csv"

# 함수 실행
filter_timeline_columns_fixed(file_path_in, file_path_out)

1. 원본 파일의 컬럼(열) 구조를 분석 중입니다...
 -> 총 50개의 컬럼 중 21개의 컬럼만 남깁니다.
 -> [확인용] 찾은 데미지 스탯 개수: 12개
 -> 예시: ['damageStats_magicDamageDone', 'damageStats_magicDamageDoneToChampions', 'damageStats_magicDamageTaken']

2. 선택된 컬럼만 추출하여 새로운 CSV로 저장합니다... (메모리 안전 모드)

3. 깔끔하게 정리가 완료되었습니다! 파일 저장 경로:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\timeline_kr_part1_all_data_filtered.csv


In [3]:
details_data = pd.read_csv(r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details_kr_part1_flattened.csv")

details_data

,assists,baronKills,bountyLevel,ch_abilityUses,ch_acesBefore15Minutes,ch_alliedJungleMonsterKills,ch_baronBuffGoldAdvantageOverThreshold,ch_baronTakedowns,ch_bountyGold,ch_buffsStolen,...,trueDamageDealtToChampions,trueDamageTaken,turretKills,turretTakedowns,turretsLost,visionScore,visionWardsBoughtInGame,wardsKilled,wardsPlaced,win
0,2,0,NaN,110,0,0,NaN,0,0.000000,0,...,0,32,1,1,1,7,0,0,5,False
1,3,0,NaN,254,0,15,NaN,0,469.360535,1,...,84,471,0,0,1,3,1,1,1,False
2,4,0,NaN,115,0,0,NaN,0,637.141968,0,...,0,59,0,1,1,9,0,1,5,False
3,5,0,NaN,60,0,0,NaN,0,359.220337,0,...,0,397,0,0,1,9,0,1,5,False
4,4,0,NaN,97,0,0,NaN,0,0.000000,0,...,299,615,0,0,1,30,7,3,17,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235075,9,0,NaN,176,0,0,1.0,0,0.000000,0,...,219,1898,0,2,1,24,6,2,6,True
235076,4,1,NaN,492,0,66,1.0,1,0.000000,3,...,3983,1106,2,4,1,29,1,2,5,True
235077,8,0,NaN,238,0,0,1.0,1,0.000000,0,...,670,957,2,3,1,29,4,3,13,True
235078,1,0,NaN,206,0,40,1.0,0,0.000000,2,...,604,1222,5,6,1,19,1,1,13,True


In [ ]:
#매치와타임 null값 확인
import pandas as pd

def check_null_values(file_path):
    """
    CSV 파일 내의 Null(결측치) 값을 검사하고,
    어떤 열에 몇 개의 Null이 있는지 위치와 비율을 출력합니다.
    """
    import os
    if not os.path.exists(file_path):
        print(f"오류: 파일을 찾을 수 없습니다.\n경로: {file_path}\n")
        return

    print(f"{"="*50}")
    print(f"🔍 파일 검사 시작: {os.path.basename(file_path)}")
    print(f"{"="*50}")
    
    try:
        # 데이터 불러오기 (메모리 절약을 위해 low_memory=False 설정)
        df = pd.read_csv(file_path, low_memory=False)
        
        # 1. 전체 데이터 크기
        total_rows = len(df)
        print(f"▶ 총 데이터 크기: {total_rows}행, {len(df.columns)}열")
        
        # 2. 결측치(Null)가 있는 열만 필터링
        null_counts = df.isnull().sum()
        null_cols = null_counts[null_counts > 0]
        
        if len(null_cols) == 0:
            print("\n✅ 이 파일에는 Null(결측치) 값이 전혀 없습니다!")
        else:
            print(f"\n⚠️ Null 값이 존재하는 열(Column)은 총 {len(null_cols)}개입니다.")
            print("▼ Null 개수 및 비율 (가장 많은 열부터 최대 20개 출력):")
            
            # 보기 편하도록 데이터프레임으로 만들어 출력
            null_df = pd.DataFrame({
                'Null 개수': null_cols,
                'Null 비율(%)': (null_cols / total_rows * 100).round(2)
            }).sort_values(by='Null 개수', ascending=False)
            
            # 주피터 환경에서 표 형태로 출력
            display(null_df.head(20))
            
        # 3. 태블로 조인(Join) 핵심 키 검사
        print("\n[태블로 연결 키 특별 검사]")
        key_cols = ['matchId', 'participantId']
        for col in key_cols:
            if col in df.columns:
                null_key_count = df[col].isnull().sum()
                if null_key_count > 0:
                    print(f" 🚨 치명적 오류: 조인 키 '{col}'에 Null 값이 {null_key_count}개 존재합니다! (반드시 수정 필요)")
                else:
                    print(f" 🟢 정상: 조인 키 '{col}'에는 Null 값이 없습니다.")
            else:
                print(f" ⚪ 주의: '{col}' 열이 파일에 아예 존재하지 않습니다.")
        print("\n")
        
    except Exception as e:
        print(f"오류 발생: {e}\n")

# ==========================================
# 실행을 위한 파일 경로 설정
# ==========================================

details_file = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details_kr_part1_flattened.csv"
timeline_file = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\timeline_kr_part1_all_filtered.csv"

# 두 파일 각각 검사 실행
check_null_values(details_file)
check_null_values(timeline_file)

🔍 파일 검사 시작: details_kr_part1_flattened.csv
▶ 총 데이터 크기: 235080행, 154열

⚠️ Null 값이 존재하는 열(Column)은 총 20개입니다.
▼ Null 개수 및 비율 (가장 많은 열부터 최대 20개 출력):


,Null 개수,Null 비율(%)
ch_hadAfkTeammate,230879,98.21
ch_highestCrowdControlScore,211636,90.03
ch_highestChampionDamage,211594,90.01
ch_highestWardKills,206840,87.99
ch_junglerKillsEarlyJungle,188067,80.00
ch_killsOnLanersEarlyJungleAsJungler,188067,80.00
ch_baronBuffGoldAdvantageOverThreshold,185895,79.08
damageDealtToEpicMonsters,158280,67.33
bountyLevel,137950,58.68
ch_controlWardTimeCoverageInRiverOrEnemyHalf,110312,46.93



[태블로 연결 키 특별 검사]
 🟢 정상: 조인 키 'matchId'에는 Null 값이 없습니다.
 🟢 정상: 조인 키 'participantId'에는 Null 값이 없습니다.


오류: 파일을 찾을 수 없습니다.
경로: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\timeline_kr_part1_all_filtered.csv



In [ ]:
#matchid null 확인
import pandas as pd
import os

def check_matchid_nulls(file_path):
    """
    지정된 파일에서 'matchId' 컬럼에 Null 값이 있는지 집중 검사합니다.
    """
    if not os.path.exists(file_path):
        print(f"오류: 지정한 경로에 파일이 없습니다.\n경로: {file_path}\n")
        return

    print(f"{'='*50}")
    print(f"🔍 [matchId] Null 집중 검사: {os.path.basename(file_path)}")
    print(f"{'='*50}")
    
    try:
        # 파일 불러오기
        df = pd.read_csv(file_path, low_memory=False)
        total_rows = len(df)
        
        # matchId 컬럼 존재 여부 확인
        if 'matchId' not in df.columns:
            print("❌ 오류: 이 파일에는 'matchId' 컬럼이 존재하지 않습니다!")
            return
            
        # matchId가 Null인 행 필터링
        null_mask = df['matchId'].isnull()
        result_df = df[null_mask]
        
        null_count = len(result_df)
        
        print(f"▶ 전체 데이터: {total_rows}행")
        
        if null_count > 0:
            print(f"🚨 경고: 'matchId'가 없는(Null) 행이 {null_count}개 발견되었습니다!")
            print("\n[문제의 데이터 미리보기 (상위 5개)]")
            
            # matchId를 맨 앞으로 가져와서 출력
            cols = ['matchId'] + [c for c in df.columns if c != 'matchId']
            display(result_df[cols].head(5))
        else:
            print("✅ 완벽합니다! 'matchId' 컬럼에 Null 값이 단 하나도 없습니다. (조인 안전)")
            
    except Exception as e:
        print(f"검사 중 오류 발생: {e}")
    print("\n")

# ==========================================
# 실행 영역: 두 파일 모두 검사
# ==========================================

details_file = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details_kr_part1_flattened.csv"
timeline_file = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\timeline_kr_part1_all_data_filtered.csv"

# 1. Details 파일 검사
check_matchid_nulls(details_file)

# 2. Timeline 파일 검사
check_matchid_nulls(timeline_file)

🔍 [matchId] Null 집중 검사: details_kr_part1_flattened.csv
▶ 전체 데이터: 235080행
✅ 완벽합니다! 'matchId' 컬럼에 Null 값이 단 하나도 없습니다. (조인 안전)


🔍 [matchId] Null 집중 검사: timeline_kr_part1_all_data_filtered.csv
▶ 전체 데이터: 6967570행
✅ 완벽합니다! 'matchId' 컬럼에 Null 값이 단 하나도 없습니다. (조인 안전)


